# **Fact Orders**

In [0]:
df= spark.sql('select * from ecommerce.silver.orders')

In [0]:
df_dimprod=spark.sql('select product_id as DimProductKey,product_id as dim_product_id from ecommerce.gold.dimproducts')
df_dimcust=spark.sql('select DimCustomerKey,customer_id as dim_customer_id from ecommerce.gold.dimcustomers')

In [0]:
df=df.join(df_dimcust,df.customer_id == df_dimcust.dim_customer_id,how='left')\
    .join(df_dimprod,df.product_id == df_dimprod.dim_product_id,how='left')

In [0]:
df=df.drop("customer_id","product_id","dim_product_id","dim_customer_id")

### **Upsert on Fact Table**

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('ecommerce.gold.FactOrders'):

    dlt_obj = DeltaTable.forName(spark,'ecommerce.gold.FactOrders')

    dlt_obj.alias('trg').merge(df.alias('src'), ' trg.order_id = src.order_id and trg.DimCustomerKey = src.DimCustomerKey and trg.DimProductKey = src.DimProductKey ')\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()
else:
    df.write.format('delta')\
        .mode('append')\
        .option('path','abfss://gold@datalakecommerce.dfs.core.windows.net/FactOrders')\
        .saveAsTable('ecommerce.gold.FactOrders')

In [0]:
sus_dlt_obj=spark.read.table('ecommerce.gold.FactOrders')

x=spark.read.format('delta').load('abfss://gold@datalakecommerce.dfs.core.windows.net/FactOrders')


print(type(dlt_obj))
print(type(sus_dlt_obj))
print(type(x))

<class 'delta.connect.tables.DeltaTable'>
<class 'pyspark.sql.connect.dataframe.DataFrame'>
<class 'pyspark.sql.connect.dataframe.DataFrame'>
